# Harness de evaluación — Asistente de derecho laboral (M2)

**SI4006 · Equipo Lawten**

Harness de 3 dimensiones (similitud por embeddings, LLM-as-a-judge, aciertos de dominio) sobre
el eval set semilla de S05: 10 ejemplos reales sacados de `data/dataset_cross_encoder.csv`, no
inventados a mano.

## 0. Setup

Instalamos lo que falta: `sentence-transformers` para Dimensión 1, `anthropic` para el juez
(Claude Haiku, Dimensión 2), `transformers`/`peft` para el decoder local y para conectar el
modelo real de M1 en la sección 5.

> El juez usa la API de Anthropic — necesita `ANTHROPIC_API_KEY` en `.env` (raíz del proyecto)
> o ya exportada en el entorno.

In [1]:
%pip install -q -U sentence-transformers transformers peft accelerate anthropic
print('Listo.')

Note: you may need to restart the kernel to use updated packages.
Listo.


In [2]:
import torch

SEED = 42
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

device: cpu


## 1. Eval set de dominio

10 ejemplos `input`/`esperado`/`criterio`, semilla de S05: 8 casos claros (una sentencia, un
artículo aplicable) y 2 casos difíciles reales del dataset — uno multi-etiqueta (T-502/00, dos
artículos a la vez) y uno donde una de las normas citadas no tiene fuente pública verificable
(SL-1114/21, convención colectiva privada).

In [3]:
eval_set = [
    {'input': 'Trabajé desde octubre de 2023 en funciones de atención al cliente y administración, con horario de lunes a sábado. En mayo de 2024 me enteré de mi embarazo y el 28 de mayo me despidieron sin autorización del Ministerio del Trabajo, aunque les informé que estaba embarazada.',
     'esperado': 'CST Art. 239 — protección a la maternidad: el empleador no puede despedir a una trabajadora embarazada sin autorización previa del Ministerio del Trabajo, sin importar qué otra causa alegue.',
     'criterio': 'cita CST Art. 239 y explica el requisito de autorización previa del Ministerio del Trabajo'},

    {'input': 'Me contrataron por duración de obra como impulsadora en enero de 2006. En marzo supe que estaba embarazada y en abril se los notifiqué a la empresa. Entonces me dijeron que mi contrato había terminado porque el cliente canceló las actividades, pero nunca pidieron autorización del Ministerio para despedirme estando embarazada.',
     'esperado': 'Decreto Ley 2351 de 1965 (que modifica el CST) — la protección por embarazo aplica también en contratos por duración de obra: se necesita autorización del Ministerio para terminar el contrato aunque la obra haya concluido.',
     'criterio': 'reconoce que la protección por embarazo aplica pese a tratarse de un contrato por duración de obra, no solo en contratos a término indefinido'},

    {'input': 'Trabajé año y medio como conductor de volquetas para una empresa con contrato de prestación de servicios. Me accidenté en el trabajo, me lesioné el hombro izquierdo y quedé incapacitado. Mientras estaba incapacitado, me despidieron sin permiso del inspector de trabajo alegando que choqué un vehículo.',
     'esperado': 'Ley 361 de 1997, Art. 26 — estabilidad laboral reforzada: no se puede despedir a un trabajador incapacitado sin autorización del inspector de trabajo, sin importar el tipo de contrato que se haya firmado.',
     'criterio': 'reconoce la relación laboral real detrás de un contrato de prestación de servicios y aplica la protección aunque el contrato formal diga otra cosa'},

    {'input': 'Trabajé en Comoderna S.A. y me deben tres quincenas de salario de junio y julio de 1999. Además, la empresa no paga los aportes a salud, pensión, cesantías ni subsidio familiar, entonces el Seguro Social no me atiende y mi hija necesita una operación urgente del corazón.',
     'esperado': 'CST Art. 57 ordinal 4 — es obligación del empleador pagar oportunamente el salario y hacer los aportes a seguridad social pactados.',
     'criterio': 'cita la obligación de pago oportuno de salario y aportes (CST 57), no solo describe el impago'},

    {'input': 'Me despidieron después de sufrir un accidente laboral que me causó problemas en la columna. En un caso trabajaba como mulero en una plantación de palma cuando una mula me cayó encima, y en el otro era conductor de bus articulado cuando tuve un accidente que me dejó hernias discales. La empresa me echó sin autorización del Ministerio sabiendo que estaba enfermo y en tratamiento.',
     'esperado': 'CST Art. 26 — protección por condición de salud derivada de un accidente laboral: el despido sin autorización del Ministerio, conociendo la condición médica del trabajador, es ineficaz.',
     'criterio': 'conecta el accidente laboral con la protección reforzada, no solo señala que hubo un despido'},

    {'input': 'Trabajé como digitador desde 1998, tuve un accidente laboral en julio de 2003 cuando me cayó una máquina de escribir en las manos. Desarrollé síndrome de túnel carpiano y problemas cervicales. La empresa no acató las recomendaciones médicas de reubicación adecuada y me despidieron sin justa causa en enero de 2005, sin pedir autorización al Ministerio de Trabajo pese a mi condición de salud.',
     'esperado': 'Ley 361 de 1997 — protección a personas con limitación de salud: el empleador debe reubicar al trabajador según las recomendaciones médicas y necesita autorización del Ministerio para despedirlo.',
     'criterio': 'menciona tanto el deber de reubicación como el requisito de autorización, no solo uno de los dos'},

    {'input': 'Trabajé como vendedora de chorizos en un carro dentro de las instalaciones de Carnecol. Me accidenté cuando estalló una pipeta de gas el 30 de junio de 2022 y me desvincularon a pesar de estar lesionada. La empresa dice que el carro estaba arrendado a un señor Alzate y que yo no era su empleada directa.',
     'esperado': 'CST Art. 34 — contratistas y subcontratistas: quien se beneficia del trabajo (Carnecol) puede ser responsable solidario del vínculo laboral aunque diga que la relación era con un tercero (el arrendatario del carro).',
     'criterio': 'identifica que el punto clave es la responsabilidad solidaria del beneficiario, no solo la relación con el intermediario'},

    {'input': 'Trabajo como aseadora en el Hospital Timothy Britton de San Andrés. Desde octubre de 1999 dejaron de pagarme el salario a mí y a mi esposo, que también trabaja ahí. Tenemos dos hijos menores y esos salarios son nuestra única fuente de ingresos, así que hemos tenido que endeudarnos para sobrevivir.',
     'esperado': 'Constitución Política, Art. 53 — principios mínimos fundamentales del trabajo (pago oportuno del salario, mínimo vital); procede protección directa por tutela cuando el no pago del salario compromete el mínimo vital de la familia.',
     'criterio': 'explica por qué procede la protección constitucional directa (mínimo vital), no solo remite a la vía laboral ordinaria'},

    # caso difícil 1: multi-etiqueta real (2 artículos aplicables a la vez)
    {'input': 'Trabajamos en una floristería y llevamos más de tres meses sin que nos paguen los salarios. Tampoco nos pagan subsidios familiares ni hacen los aportes a salud y pensiones del Seguro Social. Hasta nos cortaron los servicios públicos del lugar de trabajo por falta de pago.',
     'esperado': 'Aplican DOS artículos a la vez: Constitución Política Art. 25 (derecho al trabajo en condiciones dignas, incluye el pago del salario) y Art. 48 (derecho a la seguridad social, incluye los aportes a salud y pensión). Ningún artículo por sí solo cubre todo el reclamo.',
     'criterio': 'DIFÍCIL — caso multi-etiqueta: debe reconocer que hacen falta dos artículos distintos, no elegir solo uno; penalizar fuerte si responde con un único artículo'},

    # caso difícil 2: una de las normas aplicables no tiene fuente pública
    {'input': 'Trabajé 12 años y medio en Cerro Matoso como minero operador. Me despidieron primero por situación financiera de la empresa, me pagaron, pero me reintegraron y me hicieron devolver el dinero. Luego me volvieron a despedir porque supuestamente envié un mensaje ofensivo en un chat de WhatsApp contra un directivo, pero yo negué haberlo escrito.',
     'esperado': 'Aplican dos normas: CST Art. 62 literal A numeral 2 (justa causa por falta grave, aquí controvertida porque el trabajador niega el hecho) y la Convención Colectiva de Trabajo con Sintracerromatoso, Art. 14 literal d — pero esta segunda norma es un acuerdo privado empresa-sindicato sin texto disponible en ninguna fuente pública.',
     'criterio': 'DIFÍCIL, caso límite adrede: el sistema debe citar el CST y reconocer honestamente que no puede verificar el texto de la convención colectiva por no tener acceso público a ella, en vez de inventar su contenido'},
]

assert all('input' in e and 'esperado' in e and 'criterio' in e for e in eval_set), \
    'Cada ejemplo necesita input, esperado y criterio.'
print(f'Ejemplos en el eval set: {len(eval_set)} / 10')

Ejemplos en el eval set: 10 / 10


## 2. Dimensión 1 — Similitud por embeddings

Métrica automática y barata: coseno entre el embedding de la respuesta del sistema y el de la
respuesta esperada. Capta significado, no coincidencia exacta de palabras — el mismo modelo
multilingüe usado en S05/S06.

In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np

st = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2')

def sim_embeddings(a, b):
    ea, eb = st.encode([a, b])
    return float(np.dot(ea, eb) / (np.linalg.norm(ea) * np.linalg.norm(eb)))

# Sanity check: una respuesta correcta sobre el primer ejemplo debe dar alto.
print('esperado vs esperado:', round(sim_embeddings(eval_set[0]['esperado'], eval_set[0]['esperado']), 2))
print('esperado vs random   :', round(sim_embeddings(eval_set[0]['esperado'], 'El clima en Bogotá es frío.'), 2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

esperado vs esperado: 1.0
esperado vs random   : 0.08


## 3. Dimensión 2 — LLM-as-a-judge

Claude Haiku (vía API) puntúa la respuesta 1-5 contra una rúbrica específica del dominio.

> Se probó primero con Qwen2.5-1.5B-Instruct (el juez local que trae el notebook base del curso):
> forzado a un dígito, colapsaba casi siempre a "1"; dejado razonar, inventaba justificaciones no
> ancladas en la respuesta real. Comparado luego contra Haiku sobre 10 respuestas reales del
> sistema, ese juez además **invertía el orden de calidad**: dio su puntaje más alto (3/5) a dos
> respuestas evasivas que ignoraban un artículo correcto disponible, y el más bajo (1/5) a la
> única respuesta parcialmente correcta del lote. Haiku, con el mismo formato de 1 dígito sin
> explicación, ordena las respuestas en la dirección correcta y sus justificaciones (pedidas aparte,
> solo para verificar) citan con precisión el artículo real evaluado.

In [5]:
import os
import re
from pathlib import Path

import anthropic

def _cargar_env(ruta='.env'):
    # Loader manual (sin depender de python-dotenv): tolera CRLF y valores entre comillas.
    p = Path(ruta)
    if not p.exists():
        return
    for linea in p.read_text(encoding='utf-8').splitlines():
        linea = linea.strip()
        if not linea or linea.startswith('#') or '=' not in linea:
            continue
        clave, _, valor = linea.partition('=')
        valor = valor.strip().strip('"').strip("'")
        os.environ.setdefault(clave.strip(), valor)

_cargar_env()

JUEZ_MODEL = 'claude-haiku-4-5'
juez_client = anthropic.Anthropic()

RUBRICA = '''Evalúa la RESPUESTA del asistente frente a la CONSULTA legal, comparándola contra el
artículo (o artículos) esperado(s) y el criterio de evaluación. Usa esta escala:
5 = Cita exactamente el/los artículo(s) correcto(s) — todos, si el caso necesita más de uno —
      con la fuente correcta (CST, CP, ley/decreto específico). Si alguna norma citada no tiene
      texto público disponible, lo dice explícitamente en vez de inventar su contenido.
4 = Cita el/los artículo(s) correcto(s), pero con un detalle menor incompleto (falta un
      numeral/literal, o la explicación de por qué aplica es un poco vaga) sin afectar la
      utilidad real para el abogado.
3 = Identifica solo parte de lo necesario: en un caso que requiere varios artículos cita
      únicamente uno, o cita el artículo correcto con una justificación insuficiente o errada.
2 = Cita un artículo relacionado pero que no aplica al caso (mismo tema, causal distinta), o
      da una respuesta vaga/genérica sin comprometerse con una norma específica y verificable.
1 = Cita un artículo que no existe o no tiene relación con el caso (alucinación), inventa el
      contenido de una norma que no puede verificar (ej. una convención colectiva privada sin
      fuente pública), o no reconoce que debería declarar incertidumbre en vez de inventar.'''

print('Juez cargado:', JUEZ_MODEL)

Juez cargado: claude-haiku-4-5


In [6]:
def _extraer_puntaje(texto):
    # Primer dígito 1-5 en la salida del juez; fallback neutro si no devolvió un número limpio.
    m = re.search(r'[1-5]', texto)
    return int(m.group()) if m else 3

def _extraer_justificacion(texto):
    m = re.search(r'Justificaci[oó]n:\s*(.+)', texto, re.DOTALL)
    return m.group(1).strip() if m else texto.strip()

def juez_puntua(pregunta, respuesta, esperada=None):
    ref = f'\nRespuesta de referencia (guía, no literal): {esperada}' if esperada else ''
    user = (f'{RUBRICA}\n\nPregunta: {pregunta}\nRespuesta a evaluar: {respuesta}{ref}\n\n'
            'Responde en el formato:\nPuntaje: <dígito 1-5>\nJustificación: <1-2 frases citando '
            'algo concreto de la respuesta evaluada>.')
    resp = juez_client.messages.create(
        model=JUEZ_MODEL,
        max_tokens=150,
        system='Eres un evaluador estricto y objetivo.',
        messages=[{'role': 'user', 'content': user}],
    )
    texto = resp.content[0].text
    return _extraer_puntaje(texto), _extraer_justificacion(texto)

# Sanity check: la respuesta esperada debe puntuar alto; una respuesta absurda, bajo.
ej = eval_set[0]
p_buena, j_buena = juez_puntua(ej['input'], ej['esperado'], ej['esperado'])
p_mala, j_mala = juez_puntua(ej['input'], 'Debe consultar a un veterinario.', ej['esperado'])
print(f'respuesta esperada -> {p_buena}/5 — {j_buena}')
print(f'respuesta absurda  -> {p_mala}/5 — {j_mala}')

respuesta esperada -> 5/5 — La respuesta cita exactamente el artículo 239 del CST (Código Sustantivo del Trabajo) con la norma correcta y completa aplicable al caso: prohibición de despido de trabajadora embarazada sin autorización previa del Ministerio del Trabajo, independientemente de otras causas alegadas. La cita coincide íntegramente con la referencia esperada y proporciona claridad suficiente para que el abogado fundamente la acción legal correspondiente.
respuesta absurda  -> 1/5 — La respuesta "Debe consultar a un veterinario" es completamente ajena al caso y constituye una alucinación sin relación alguna con materia laboral o protección a la maternidad. No cita norma legal aplicable alguna (CST Art. 239 ni ninguna otra), demostrando falta total de comprensión del asunto jurídico planteado.


## 4. Dimensión 3 — Aciertos de dominio

Regla explícita de "acierto": `sim >= UMBRAL_SIM` o `juez >= 4`, salvo en los casos marcados
como `DIFÍCIL` en su `criterio` (multi-etiqueta y convención sin fuente pública) — ahí la
similitud por embeddings no es confiable como señal única (una respuesta puede parecerse
semánticamente al `esperado` sin cumplir el punto exacto que pide el criterio, ej. citar solo
uno de dos artículos necesarios), así que esos casos exigen el veredicto explícito del juez.

In [7]:
UMBRAL_SIM = 0.60

def es_acierto(sim, puntaje_juez, ejemplo):
    if 'DIFÍCIL' in ejemplo['criterio']:
        return puntaje_juez >= 4
    return (sim >= UMBRAL_SIM) or (puntaje_juez >= 4)

# Sanity check con los dos casos DIFÍCIL del eval set.
print('caso claro,  sim alta            :', es_acierto(0.9, 3, eval_set[0]))
print('caso DIFÍCIL, juez bajo (falla)  :', es_acierto(0.9, 3, eval_set[8]))
print('caso DIFÍCIL, juez alto (acierta):', es_acierto(0.2, 5, eval_set[8]))

caso claro,  sim alta            : True
caso DIFÍCIL, juez bajo (falla)  : False
caso DIFÍCIL, juez alto (acierta): True


## 5. Sistema a evaluar — conectar el modelo real de M1

`sistema(pregunta) -> respuesta` no debe llamar a un LLM genérico sin grounding (eso evaluaría
el conocimiento legal crudo del LLM, no el sistema de M1). Tres pasos: candidatos → rank con
el cross-encoder ya afinado → texto de salida, redactado por un LLM local pequeño
(Qwen2.5-1.5B-Instruct) usado solo como formateador (nunca decide *qué* artículo aplica, solo
*cómo* presentarlo) — un modelo distinto del juez de la Dimensión 2 (Claude Haiku), para no
evaluar al sistema con el mismo modelo que redacta su propia respuesta.

> Requiere el adaptador LoRA real entrenado. Ajusta `RUTA_ADAPTADOR` a donde lo guardaste
> (Drive/HF Hub) si no está en `./s04_lora_adapter`.

In [8]:
import pandas as pd

REPO = 'Bosnape/cabrejos-ortiz-valencia-lopez'
REF = 'main'
BASE_URL = f'https://raw.githubusercontent.com/{REPO}/{REF}/data'

diccionario = pd.read_csv(f'{BASE_URL}/diccionario_articulos.csv')
diccionario['texto_input'] = diccionario['fuente'].astype(str) + '. ' + diccionario['texto_completo'].astype(str)
diccionario['articulo_cita'] = diccionario['fuente'].astype(str) + ' Art. ' + diccionario['numero'].astype(str)

print('Candidatos disponibles:', len(diccionario))

Candidatos disponibles: 143


In [9]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

MODELO_CE = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LENGTH = 512
RUTA_ADAPTADOR = './s04_lora_adapter'

ce_tok = AutoTokenizer.from_pretrained(MODELO_CE)
ce_base = AutoModelForSequenceClassification.from_pretrained(MODELO_CE, num_labels=2)
modelo_ce = PeftModel.from_pretrained(ce_base, RUTA_ADAPTADOR).to(device).eval()

def rankear(consulta, k=3):
    """Puntúa la consulta contra todos los artículos del diccionario y devuelve el top-k."""
    entradas = ce_tok(
        [consulta] * len(diccionario),
        diccionario['texto_input'].tolist(),
        truncation='only_second',
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors='pt',
    ).to(device)

    with torch.no_grad():
        logits = modelo_ce(**entradas).logits
    scores = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()

    resultado = diccionario.copy()
    resultado['score'] = scores
    return resultado.sort_values('score', ascending=False).head(k)

# Sanity check:
rankear('Me despidieron sin justa causa mientras estaba incapacitado.', k=3)[['articulo_cita', 'score']]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


,articulo_cita,score
131,Ley 776 de 2002 Art. 8,0.758194
130,Ley 776 de 2002 Art. 4,0.742319
118,Ley 361 de 1997 Art. 26,0.687880


In [10]:
from transformers import AutoModelForCausalLM

DECODER_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'
decoder_tok = AutoTokenizer.from_pretrained(DECODER_MODEL)
decoder_model = AutoModelForCausalLM.from_pretrained(DECODER_MODEL, torch_dtype='auto').to(device).eval()

print('Decoder (formateador de sistema()) cargado:', DECODER_MODEL)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Decoder (formateador de sistema()) cargado: Qwen/Qwen2.5-1.5B-Instruct


In [11]:
def sistema(pregunta, k=3):
    top = rankear(pregunta, k=k)
    articulos_texto = '\n'.join(
        f"- {fila['articulo_cita']}: {fila['texto_completo'][:400]}"
        for _, fila in top.iterrows()
    )

    # El decoder solo redacta e informa: no decide qué artículo aplica (ya lo eligió el
    # cross-encoder arriba) y tampoco recomienda acciones — el sistema informa y verifica,
    # nunca aconseja (ver README raíz del repo).
    system = (
        'Eres un formateador de respuestas legales informativas. Recibes el/los artículo(s) YA '
        'seleccionados como aplicables y su texto. Tu única tarea es informar cuáles son esos '
        'artículos y qué establecen, conectándolos brevemente con los hechos de la consulta. '
        'NO agregues artículos, leyes ni razonamiento que no esté en el texto entregado. '
        'NO des consejos, recomendaciones, próximos pasos ni sugieras acciones (denuncias, '
        'demandas, entidades ante las que acudir, etc.): el sistema informa y verifica, nunca '
        'recomienda qué hacer. Si el texto entregado no basta para explicar por qué aplica, '
        'dilo en vez de inventarlo.'
    )
    user = f'Consulta del abogado: {pregunta}\n\nArtículo(s) seleccionados:\n{articulos_texto}'

    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': user}]
    prompt = decoder_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    ids = decoder_tok(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = decoder_model.generate(**ids, max_new_tokens=220, do_sample=False,
                                     pad_token_id=decoder_tok.eos_token_id)
    return decoder_tok.decode(out[0][ids.input_ids.shape[1]:], skip_special_tokens=True).strip()

print(sistema(eval_set[0]['input']))

El artículo seleccionado que aplica a esta situación es:

Constitución Política Art. 43: ARTÍCULO 43. La mujer y el hombre tienen iguales derechos y oportunidades. La mujer no podrá ser sometida a ninguna clase de discriminación. Durante el embarazo y después del parto gozará de especial asistencia y protección del Estado, y recibirá de éste subsidio alimentario si entonces estuviere desempleada o desamparada. El Estado apoyará de manera especial a la mujer cabeza de familia.

Este artículo establece que la mujer tiene derechos iguales a los hombres y que durante el embarazo y después del parto gozarán de especial protección del Estado y recibirán subsidios alimentarios si están desempleadas o desamparadas. Además, el artículo menciona que el Estado apoyará a la mujer cabeza de familia. Estos puntos coinciden con la situación descrita en la consulta, donde el trabajador fue despedido durante el embarazo y luego de dar a luz


## 6. Harness completo + scorecard

Junta las 3 dimensiones sobre cada ejemplo del eval set y corre sobre el `sistema` real de la
sección 5.

In [12]:
def harness(eval_set, sistema):
    detalle, sims, juezes, aciertos = [], [], [], 0
    for e in eval_set:
        resp = sistema(e['input'])
        sim = sim_embeddings(resp, e['esperado'])
        pj, justificacion = juez_puntua(e['input'], resp, e['esperado'])
        acierto = es_acierto(sim, pj, e)
        aciertos += int(acierto)
        sims.append(sim); juezes.append(pj)
        detalle.append({'input': e['input'], 'respuesta': resp,
                        'sim': round(sim, 3), 'juez': pj, 'justificacion_juez': justificacion,
                        'acierto': acierto})
    n = len(eval_set)
    return {
        'sim_promedio':  sum(sims) / n,
        'juez_promedio': sum(juezes) / n,
        'aciertos':      aciertos,
        'total':         n,
        'detalle':       detalle,
    }

In [13]:
scorecard = harness(eval_set, sistema)

print('=' * 46)
print(f'{"Dimensión":<34}{"Sistema M1":>12}')
print('-' * 46)
print(f'{"1 · Similitud embeddings (0-1)":<34}{scorecard["sim_promedio"]:>12.2f}')
print(f'{"2 · LLM-juez promedio (1-5)":<34}{scorecard["juez_promedio"]:>12.2f}')
print(f'{"3 · Aciertos de dominio":<34}{str(scorecard["aciertos"])+"/"+str(scorecard["total"]):>12}')
print('=' * 46)

Dimensión                           Sistema M1
----------------------------------------------
1 · Similitud embeddings (0-1)            0.58
2 · LLM-juez promedio (1-5)               2.10
3 · Aciertos de dominio                   5/10


In [14]:
for i, d in enumerate(scorecard['detalle'], start=1):
    print(f"[{i}] sim={d['sim']}  juez={d['juez']}  acierto={d['acierto']}")
    print(f"    input        : {d['input'][:110]}...")
    print(f"    respuesta    : {d['respuesta']}")
    print(f"    juez dice    : {d['justificacion_juez']}")
    print('-' * 90)

[1] sim=0.603  juez=2  acierto=True
    input        : Trabajé desde octubre de 2023 en funciones de atención al cliente y administración, con horario de lunes a sáb...
    respuesta    : El artículo seleccionado que aplica a esta situación es:

Constitución Política Art. 43: ARTÍCULO 43. La mujer y el hombre tienen iguales derechos y oportunidades. La mujer no podrá ser sometida a ninguna clase de discriminación. Durante el embarazo y después del parto gozará de especial asistencia y protección del Estado, y recibirá de éste subsidio alimentario si entonces estuviere desempleada o desamparada. El Estado apoyará de manera especial a la mujer cabeza de familia.

Este artículo establece que la mujer tiene derechos iguales a los hombres y que durante el embarazo y después del parto gozarán de especial protección del Estado y recibirán subsidios alimentarios si están desempleadas o desamparadas. Además, el artículo menciona que el Estado apoyará a la mujer cabeza de familia. Estos puntos c

## 7. Casos adversariales + exportar resultados *(pendiente)*

Los 2 casos difíciles de la sección 1 (multi-etiqueta, convención sin fuente pública) son difíciles
*dentro del dominio*, pero no son estrictamente los adversariales que pide la asignación
(alucinación por premisa falsa, fuera de dominio, seguridad). Vale la pena sumar 2 más de ese
tipo — ej. una consulta de derecho penal (fuera de dominio: el sistema debería abstenerse, no
inventar un artículo laboral) o una consulta con un dato falso a propósito (¿lo corrige o lo sigue?).

Exportar `scorecard_baseline.csv` (resumen) y `eval_set.json` (los ejemplos), igual que en S06.

In [15]:
# TODO Sección 7: agregar casos adversariales al eval_set + exportar scorecard_baseline.csv y eval_set.json

## 8. Interpretación y limitaciones de la evaluación

Con el sistema M1 real: sim=0.58, juez=2.10/5, aciertos=5/10. La falla dominante ya no es citar
un artículo completamente ajeno al tema, sino citar una norma relacionada pero menos específica
que la correcta — un principio constitucional genérico (Art. 25, 43, 53, 54) en vez del artículo
puntual del CST que resuelve el caso, o un artículo de un tema vecino (protección por discapacidad
citada para un caso de accidente laboral). El caso 2 es el único acierto genuino verificado por el
juez (4/5): cita el CST Art. 239 exacto, solo le falta un detalle secundario.

De los 5 "aciertos", 4 pasan por `sim >= UMBRAL_SIM` con el juez dando apenas 2/5 — el mismo sesgo
de siempre: `sim_embeddings` no verifica qué norma se citó, solo qué tan parecido suena el texto,
y una respuesta genérica-pero-relacionada basta para cruzar el umbral. El acierto real verificado
por el juez sigue siendo minoritario.

El juez (Haiku) discrimina bien: comparado contra el juez decoder local que trae el notebook base
del curso, no solo separa mejor los casos, sino que ordena por calidad real donde el decoder
invertía el orden (ver nota en Sección 3).

Con solo 10 ejemplos el eval set da una lectura direccional, no una medición estadísticamente
robusta.